# Variance Threshold
        **unsupervised › feature_selection** · notebook `variance_threshold`

        **Kapan dipakai:** Membuang fitur konstan/hampir konstan. Langkah pembersihan awal, murah.

        **Jebakan umum:** Tidak mempertimbangkan target: fitur noise bervarians tinggi TIDAK terbuang (lihat hasil).

**Data demo:** Breast cancer + 30 fitur noise + 4 duplikat. Kita tahu mana fitur palsu (`noise_*` acak murni dan `dup_*` duplikat), sehingga kualitas seleksi bisa diukur secara objektif.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/breast_cancer_noisy.csv"; TARGET = "diagnosis"
K = 8                  # jumlah fitur target (untuk metode yang membutuhkannya)
N_SPLITS, SEED = 3, 42
MODEL_KEY = "variance_threshold"
OUT = out_dir("unsupervised", "feature_selection", MODEL_KEY)
set_seed(SEED)

## 2. Data

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
df = pd.read_csv(PATH); names = [c for c in df.columns if c != TARGET]
X = df[names].to_numpy(float); y = LabelEncoder().fit_transform(df[TARGET])
is_noise = lambda n: n.startswith("noise_"); is_dup = lambda n: n.startswith("dup_")
print(X.shape, f"| {sum(map(is_noise, names))} fitur noise, {sum(map(is_dup, names))} duplikat, {sum(not is_noise(n) and not is_dup(n) for n in names)} fitur asli")

## 3. Metode seleksi
Bentuk seragam: `select(Xtr, ytr, names)` mengembalikan `(daftar fitur terpilih, ranking atau None)`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

def select(Xtr, ytr, names):
    Xs = StandardScaler().fit_transform(Xtr)
    sel = VarianceThreshold(threshold=0.2).fit(Xs)
    return [n for n, k in zip(names, sel.get_support()) if k], None

## 4. Seleksi pada seluruh data & kualitasnya
Idealnya: fitur asli terpilih, noise dibuang, duplikat tidak ganda.

In [ ]:
import time; t0 = time.time(); sel, rank = select(X, y, names); sec = time.time() - t0
n_noise, n_dup = sum(map(is_noise, sel)), sum(map(is_dup, sel)); n_real = len(sel) - n_noise - n_dup
print(f"terpilih {len(sel)} fitur dalam {sec:.1f} dtk: {n_real} asli, {n_dup} duplikat, {n_noise} NOISE (salah)")
print("fitur terpilih:", sel[:12], "..." if len(sel) > 12 else "")
if rank is not None: display(rank.head(10).round(4).to_frame("skor"))

## 5. Evaluasi JUJUR: seleksi di dalam cross-validation
Seleksi harus dilakukan hanya pada data training tiap fold. Seleksi di seluruh data lalu CV = **kebocoran** yang membuat skor terlalu bagus.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
idx = {n: i for i, n in enumerate(names)}
mk = {"logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
      "lightgbm": lambda: LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=7, min_child_samples=5, random_state=SEED, verbose=-1)}
res, sets = {k: {"semua fitur": [], "terseleksi": []} for k in mk}, []
for tr, va in StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(X, y):
    s_, _ = select(X[tr], y[tr], names); sets.append(set(s_)); cols = [idx[n] for n in s_]
    for k, f in mk.items():
        res[k]["semua fitur"].append(f1_score(y[va], f().fit(X[tr], y[tr]).predict(X[va]), average="macro"))
        res[k]["terseleksi"].append(f1_score(y[va], f().fit(X[tr][:, cols], y[tr]).predict(X[va][:, cols]), average="macro"))
tbl = pd.DataFrame({k: {kk: np.mean(v) for kk, v in d.items()} for k, d in res.items()}); display(tbl.round(4))
jac = np.mean([len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:]]); print(f"jumlah fitur rata-rata: {np.mean([len(s) for s in sets]):.1f} | stabilitas antar fold (Jaccard): {jac:.2f}")

In [ ]:
save_json({"selected": sel, "n_real": n_real, "n_dup": n_dup, "n_noise": n_noise, "cv_f1": tbl.to_dict(), "jaccard_stability": float(jac), "seconds": sec}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Boosting dan model regularisasi sering tahan terhadap fitur noise; seleksi paling bermanfaat untuk kecepatan, interpretabilitas, dan model sensitif (KNN, SVM).
- Stabilitas rendah (Jaccard kecil) = hasil seleksi tidak dapat dipercaya; naikkan data atau pilih metode yang lebih stabil.